# CSPF — alignment fix (Phase 2 originals + Phase 3 re-run)
**Bug fixed:** Phase 2 deduplicated original texts by `review_idx` alone, but `review_idx` restarts at 0 in every dataset. Each original therefore received the prediction of review #k of the *first* dataset. Perturbed-text predictions were correct; original-text predictions, `*_flip`, RS, DR, IRR, Acc_o and everything derived from them were not.
**What this notebook does:** keeps the (correct) perturbed predictions in `ALL_MODELS_combined.csv`, re-predicts only the unique originals with composite key `source_dataset::review_idx`, recomputes flips, then re-runs Phase 3 (tables 01–15). Runtime: a few minutes per model on a T4 (unique originals only).
Run cells in order. Set the three paths in cell 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ── EDIT THESE ──────────────────────────────────────────────────────────
COMBINED_CSV = "/content/drive/MyDrive/postdoc-papers/paper2/result of labeling/ALL_MODELS_combined.csv"
FIXED_CSV    = "/content/drive/MyDrive/postdoc-papers/paper2/ALL_MODELS_combined_FIXED.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/postdoc-papers/paper2/orig_preds"   # cache on Drive (survives disconnects)
# ────────────────────────────────────────────────────────────────────────
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Colab already has numpy/pandas/scipy/sklearn/statsmodels/transformers. Do NOT upgrade them (breaks numpy).
!pip -q install sentencepiece emoji

In [ ]:
import os, gc, warnings, zipfile
import pandas as pd, numpy as np, torch
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          T5ForConditionalGeneration, T5Tokenizer)
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE=="cuda" else 8
MAX_LENGTH = 128
print("device:", DEVICE)
# CELL 3 — Model registry
# ═══════════════════════════════════════════════════════════════════════════════
MODEL_REGISTRY = [
    ("RoBERTa",    "Etelis/Sentiment140_roBERTa_5E",                    "seq_cls"),
    ("XLNet",      "Etelis/Sentiment140_XLNET_5E",                      "seq_cls"),
    ("ELECTRA",    "Etelis/Sentiment140_ELECTRA_5E",                    "seq_cls"),
    ("DistilBERT", "Etelis/Sentiment140_DistilBERT_5E",                 "seq_cls"),
    ("ALBERT",     "Etelis/Sentiment140_ALBERT_5E",                     "seq_cls"),
    ("BERT",       "Etelis/Sentiment140_BERT_5E",                       "seq_cls"),
    ("GPT2",       "ashok2216/gpt2-amazon-sentiment-classifier-V1.0",   "seq_cls"),
    ("T5",         "michelecafagna26/t5-base-finetuned-sst2-sentiment", "seq2seq_t5"),
]


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Label helpers
# ═══════════════════════════════════════════════════════════════════════════════

def normalise_seq_cls(raw: str) -> str:
    """Standard label normaliser for seq_cls models."""
    s = str(raw).strip().lower()
    lmap = {
        "positive": "positive", "negative": "negative",
        "pos":      "positive", "neg":      "negative",
        "1":        "positive", "0":        "negative",
        "label_1":  "positive", "label_0":  "negative",
        "label_2":  "positive",
        "4":        "positive",
        "positive sentiment": "positive",
        "negative sentiment": "negative",
        "positive.": "positive", "negative.": "negative",
    }
    if s in lmap:
        return lmap[s]
    if "pos" in s: return "positive"
    if "neg" in s: return "negative"
    try:
        return "positive" if int(s) >= 1 else "negative"
    except ValueError:
        pass
    return "positive"


def decode_t5(raw: str) -> str:
    """
    T5-specific decoder (michelecafagna26/t5-base-finetuned-sst2-sentiment).

    Per HF model card:
      prefix = "sentiment: <text>"
      output = 'p' (positive) or 'n' (negative)

    Fallbacks handle edge cases where full words appear.
    """
    s = str(raw).strip().lower()
    if not s:
        return "positive"
    # Primary: single-char output (documented model behavior)
    if s == "p": return "positive"
    if s == "n": return "negative"
    # Fallback: substring search
    if "neg" in s: return "negative"
    if "pos" in s: return "positive"
    # Numeric
    if s in ("0", "false"): return "negative"
    if s in ("1", "true"):  return "positive"
    return "positive"


def get_id2label(model) -> dict:
    cfg = model.config
    if hasattr(cfg, "id2label") and cfg.id2label:
        return {int(k): str(v) for k, v in cfg.id2label.items()}
    return {0: "negative", 1: "positive"}


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Model loading helpers
# ═══════════════════════════════════════════════════════════════════════════════

def load_seq_cls_safe(model_id: str, device: str):
    """
    Load a seq_cls model trying multiple strategies.
    Loads in float32, then casts to float16 on GPU.
    """
    strategies = [
        lambda: AutoModelForSequenceClassification.from_pretrained(model_id),
        lambda: AutoModelForSequenceClassification.from_pretrained(
                    model_id, trust_remote_code=True),
        lambda: AutoModelForSequenceClassification.from_pretrained(
                    model_id, ignore_mismatched_sizes=True),
    ]
    last_err = None
    for i, strategy in enumerate(strategies, 1):
        try:
            model = strategy().to(device)
            if device == "cuda":
                model = model.half()
            print(f"  Model loaded (strategy {i})")
            return model
        except Exception as e:
            last_err = e
            if i < len(strategies):
                print(f"  Strategy {i} failed: {type(e).__name__}: {str(e)[:80]}")
                print(f"  Trying strategy {i+1} …")
    raise RuntimeError(f"All loading strategies failed. Last error: {last_err}")


def load_tweeberta(model_id: str, device: str):
    """
    TweeBERTa uses TensorFlow weights — must use from_tf=True.
    """
    try:
        import tensorflow as tf  # noqa: F401 — required for TF weight loading
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            from_tf=True,
            trust_remote_code=True
        ).to(device)
        if device == "cuda":
            model = model.half()
        print("  ✓ TweeBERTa loaded (TF → PT)")
        return model
    except Exception as e:
        print(f"  ✗ TweeBERTa loading failed: {e}")
        print("  Fix: pip install tensorflow >= 2.13")
        return None


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Inference functions
# ═══════════════════════════════════════════════════════════════════════════════

def predict_seq_cls(texts: list, tokenizer, model, id2label: dict,
                    batch_size: int = BATCH_SIZE,
                    desc: str = "Predicting") -> list:
    """Standard seq_cls inference with float16/32 mismatch handling."""
    model.eval()
    model_dtype = next(model.parameters()).dtype
    results = []
    for i in tqdm(range(0, len(texts), batch_size),
                  total=(len(texts) + batch_size - 1) // batch_size,
                  desc=desc, unit="batch", leave=True):
        batch = texts[i: i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            try:
                logits = model(**enc).logits
            except RuntimeError as e:
                if "Half" in str(e) or "Float" in str(e) or "dtype" in str(e):
                    enc = {k: v.to(model_dtype) if v.is_floating_point() else v
                           for k, v in enc.items()}
                    logits = model(**enc).logits
                else:
                    raise
        for p in logits.argmax(dim=-1).cpu().tolist():
            results.append(normalise_seq_cls(id2label.get(p, str(p))))
    return results


def predict_t5(texts: list, tokenizer, model,
               batch_size: int = BATCH_SIZE,
               desc: str = "T5 Predicting") -> list:
    """
    T5 inference with correct prefix "sentiment: " per model card.
    Output is 'p' or 'n' → decoded via decode_t5().
    """
    model.eval()
    results = []
    first = True
    for i in tqdm(range(0, len(texts), batch_size),
                  total=(len(texts) + batch_size - 1) // batch_size,
                  desc=desc, unit="batch", leave=True):
        batch = texts[i: i + batch_size]
        prefixed = ["sentiment: " + t for t in batch]   # ← required prefix
        enc = tokenizer(prefixed, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out_ids = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=3,   # single char 'p' or 'n'
            )
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)

        if first:
            print(f"\n  [T5 DEBUG] First 5 raw outputs:")
            for raw in decoded[:5]:
                print(f"    '{raw}'  →  '{decode_t5(raw)}'")
            first = False

        results.extend(decode_t5(d) for d in decoded)
    return results


# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ── load corrupted file, build composite key, unique originals ───────────
df = pd.read_csv(COMBINED_CSV, low_memory=False)
df["review_key"] = df["source_dataset"].astype(str) + "::" + df["review_idx"].astype(str)
print(len(df), "pairs |", df.review_idx.nunique(), "raw review_idx |", df.review_key.nunique(), "composite keys")
uo = df[["review_key","original_review","sentiment","source_dataset"]].drop_duplicates("review_key").reset_index(drop=True)
# the original text must be identical for a key
assert (df.groupby("review_key").original_review.nunique()==1).all()
print("unique originals:", len(uo))
MODELS = [m for m,_,_ in MODEL_REGISTRY if f"{m}_pert_label" in df.columns]
print("models present:", MODELS)

In [ ]:
# ── re-predict ORIGINALS only (cached per model) ──────────────────────────
reg = {m:(i,k) for m,i,k in MODEL_REGISTRY}
texts = uo.original_review.astype(str).tolist()
for m in MODELS:
    ck = f"{OUTPUT_DIR}/{m}_orig.csv"
    if os.path.exists(ck): print("cached", m); continue
    mid, cls = reg[m]
    print("==", m, mid)
    tok = T5Tokenizer.from_pretrained(mid) if cls=="seq2seq_t5" else AutoTokenizer.from_pretrained(mid)
    if cls=="seq2seq_t5":
        model = T5ForConditionalGeneration.from_pretrained(mid).to(DEVICE)
        if DEVICE=="cuda": model = model.half()
    else:
        model = load_seq_cls_safe(mid, DEVICE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        if cls=="seq_cls": model.config.pad_token_id = tok.eos_token_id
    if cls=="seq2seq_t5": lab = predict_t5(texts, tok, model, desc=m)
    else: lab = predict_seq_cls(texts, tok, model, get_id2label(model), desc=m)
    pd.DataFrame({"review_key":uo.review_key, "orig_label":lab}).to_csv(ck, index=False)
    del model, tok; gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()

In [ ]:
# ── merge corrected originals, recompute flips, SANITY GATE, save ─────────
old_orig = {m: df[f"{m}_orig_label"].copy() for m in MODELS}
for m in MODELS:
    mp = pd.read_csv(f"{OUTPUT_DIR}/{m}_orig.csv").set_index("review_key")["orig_label"]
    df[f"{m}_orig_label"] = df["review_key"].map(mp)
    assert df[f"{m}_orig_label"].notna().all()
    df[f"{m}_flip"] = (df[f"{m}_orig_label"] != df[f"{m}_pert_label"]).astype(int)

rows=[]
for m in MODELS:
    u = df.drop_duplicates("review_key")
    acc_o_new = (u[f"{m}_orig_label"]==u.sentiment).mean()
    acc_o_old = (old_orig[m].loc[u.index]==u.sentiment).mean()
    acc_p = (df[f"{m}_pert_label"]==df.sentiment).mean()
    acc_o_pair = (df[f"{m}_orig_label"]==df.sentiment).mean()
    rows.append(dict(model=m, acc_orig_unique_NEW=acc_o_new, acc_orig_unique_OLD=acc_o_old,
                     acc_orig_pairweighted_NEW=acc_o_pair, acc_pert=acc_p,
                     RS_NEW=df[f"{m}_flip"].mean(),
                     agree_old_new_orig=(old_orig[m]==df[f"{m}_orig_label"]).mean()))
chk = pd.DataFrame(rows).round(4); print(chk.to_string(index=False))
# gate: with aligned labels, pair-weighted original accuracy should be close to perturbed accuracy
gap = (chk.acc_pert - chk.acc_orig_pairweighted_NEW).abs().max()
print("max |Acc_p - Acc_o| over models:", round(gap,4))
assert gap < 0.08, "STILL MISALIGNED — send me this printed table"
df.to_csv(FIXED_CSV, index=False); print("saved", FIXED_CSV)
chk.to_csv("alignment_check.csv", index=False)
u = df.drop_duplicates("review_key")
per_ds = pd.DataFrame({m:(u[f"{m}_orig_label"]==u.sentiment) for m in MODELS}).groupby(u.source_dataset).mean()
per_ds["mean_acc"] = per_ds.mean(axis=1); per_ds = per_ds.round(3)
print(per_ds.to_string()); per_ds.to_csv("per_dataset_orig_accuracy.csv")


## Optional: add modern fine-tuned models
Predicts originals + perturbed texts with extra off-the-shelf fine-tuned checkpoints (Twitter-trained and review-trained, so you also get an in-domain vs out-of-domain comparison). Results are cached on Drive per model; a model that fails to load is skipped. Delete names from `RUN_EXTRA` to skip. ~10–25 min per model on a T4.

In [ ]:
# ══ EXTRA MODELS (optional) — modern, already fine-tuned checkpoints ══════════
# Each model is skipped automatically if it cannot be downloaded/loaded.
# 3-class models (neg/neu/pos): the argmax is taken over the NEGATIVE and POSITIVE logits only,
# because the gold labels are binary. Say this in the paper.
EXTRA_MODELS = [
 # name,              HF id,                                              training domain (report in the paper)
 ("SiEBERT-large",    "siebert/sentiment-roberta-large-english",          "mixed reviews/tweets (15 datasets)"),
 ("TwRoBERTa-latest", "cardiffnlp/twitter-roberta-base-sentiment-latest", "Twitter (124M tweets, TweetEval)"),
 ("BERTweet",         "finiteautomata/bertweet-base-sentiment-analysis",  "Twitter (SemEval-2017)"),
 ("XLM-R-Twitter",    "cardiffnlp/twitter-xlm-roberta-base-sentiment",    "Twitter, multilingual"),
 ("DistilBERT-SST2",  "distilbert-base-uncased-finetuned-sst-2-english",  "movie reviews (SST-2)"),
 ("RoBERTa-SST2",     "textattack/roberta-base-SST-2",                    "movie reviews (SST-2)"),
 ("DistilBERT-IMDb",  "lvwerra/distilbert-imdb",                          "movie reviews (IMDb)"),
 ("ModernBERT-sent",  "clapAI/modernBERT-base-multilingual-sentiment",    "multilingual sentiment (ModernBERT)"),
]
RUN_EXTRA = [m[0] for m in EXTRA_MODELS]        # delete names here to skip models
CACHE_DIR = os.path.join(os.path.dirname(FIXED_CSV), "model_cache"); os.makedirs(CACHE_DIR, exist_ok=True)
EXTRA_BS, EXTRA_ML = 128, 128

def pos_neg_indices(id2label):
    items = {int(k): str(v).lower().strip() for k, v in id2label.items()}
    neg = [k for k, v in items.items() if v.startswith("neg") or v in ("label_negative",)]
    pos = [k for k, v in items.items() if v.startswith("pos") or v in ("label_positive",)]
    if not (neg and pos):                                   # generic LABEL_i names
        n = len(items)
        if n == 2:   neg, pos = [0], [1]
        elif n == 3: neg, pos = [0], [2]                    # cardiffnlp convention: 0 neg, 1 neu, 2 pos
        else: raise ValueError(f"cannot map labels {items}")
    return neg[0], pos[0]

@torch.no_grad()
def predict_pos(texts, tok, model, neg_i, pos_i):
    """returns int8 array: 1 = positive, 0 = negative (argmax over the two polarity logits only)."""
    order = np.argsort([len(t) for t in texts])             # length-sorted batches = much faster
    out = np.zeros(len(texts), dtype=np.int8)
    for s in tqdm(range(0, len(texts), EXTRA_BS), leave=False):
        ix = order[s:s+EXTRA_BS]
        enc = tok([texts[i] for i in ix], padding=True, truncation=True, max_length=EXTRA_ML, return_tensors="pt").to(DEVICE)
        lg = model(**enc).logits.float()
        out[ix] = (lg[:, pos_i] > lg[:, neg_i]).cpu().numpy().astype(np.int8)
    return out

# unique texts shared by all models
pert_codes, pert_uni = pd.factorize(df["perturbed_review"].astype(str))
orig_texts = uo.original_review.astype(str).tolist()          # aligned with uo.review_key
key2row = pd.Series(np.arange(len(uo)), index=uo.review_key)
orig_row = key2row.loc[df.review_key].values
print(len(pert_uni), "unique perturbed texts;", len(orig_texts), "unique originals")

added = []
for name, mid, dom in EXTRA_MODELS:
    if name not in RUN_EXTRA: continue
    fp, fo = f"{CACHE_DIR}/{name}_pert.npy", f"{CACHE_DIR}/{name}_orig.npy"
    try:
        if os.path.exists(fp) and os.path.exists(fo):
            p_lab, o_lab = np.load(fp), np.load(fo); print("cached", name)
        else:
            print("==", name, mid, "|", dom)
            tok = AutoTokenizer.from_pretrained(mid)
            model = AutoModelForSequenceClassification.from_pretrained(mid).to(DEVICE).eval()
            if DEVICE == "cuda": model = model.half()
            neg_i, pos_i = pos_neg_indices(model.config.id2label); print("  neg/pos logit idx:", neg_i, pos_i, model.config.id2label)
            o_lab = predict_pos(orig_texts, tok, model, neg_i, pos_i)
            p_lab = predict_pos(pert_uni.tolist(), tok, model, neg_i, pos_i)
            np.save(fp, p_lab); np.save(fo, o_lab)
            del model, tok; gc.collect(); torch.cuda.empty_cache()
        to = np.where(o_lab == 1, "positive", "negative")
        tp = np.where(p_lab == 1, "positive", "negative")
        df[f"{name}_orig_label"] = to[orig_row]
        df[f"{name}_pert_label"] = tp[pert_codes]
        df[f"{name}_flip"] = (df[f"{name}_orig_label"] != df[f"{name}_pert_label"]).astype(int)
        added.append(name)
        print(f"  {name}: acc_orig={(df[f'{name}_orig_label']==df.sentiment).mean():.3f}  "
              f"acc_pert={(df[f'{name}_pert_label']==df.sentiment).mean():.3f}  RS={df[f'{name}_flip'].mean():.4f}")
    except Exception as e:
        print(f"  ✗ {name} skipped: {type(e).__name__}: {str(e)[:150]}")
print("added:", added)
pd.DataFrame(EXTRA_MODELS, columns=["model","hf_id","training_domain"]).to_csv("extra_models_registry.csv", index=False)
df.to_csv(FIXED_CSV, index=False); print("re-saved", FIXED_CSV)


## Phase 3 re-run (your original code, only the accuracy join changed to the composite key)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  CSPF — Phase 3: Comprehensive Evaluation Pipeline  (FINAL v2)              ║
║  Google Colab Compatible                                                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  INPUT  : ALL_MODELS_combined.csv  (output of Phase 2)                      ║
║  OUTPUT : cspf_evaluation_results/                                           ║
║    01  flip_rate_matrix.csv          FR(family × severity × model) + 95% CI ║
║    02  model_robustness_scores.csv   RS, RS_c, DR, IRR, acc + Bootstrap CI  ║
║    03  severity_monotonicity.csv     Page's L (correct), Spearman ρ         ║
║    04  position_analysis.csv         FR by position + Kruskal-Wallis        ║
║    05  dataset_vulnerability.csv     DVI per dataset                        ║
║    06a kruskal_wallis.csv            H, df, p per family                    ║
║    06b dunn_posthoc.csv              Bonferroni-corrected pairwise z / p    ║
║    07  pir_tensor.pkl                Full 4D tensor [12×3×3×9]              ║
║    08  effect_sizes.csv              Cohen's h + 95% CI (bootstrap)         ║
║    09  semantic_vs_flip.csv          r_pb + Spearman ρ + 95% CI             ║
║    10  severity_drift.csv            Metric means per (metric, fam, sev)    ║
║    11  linguistic_target_fr.csv      FR per linguistic target type           ║
║    12  cross_dataset_fr.csv          FR heatmap (family × dataset)          ║
║    13  agreement_matrix.csv          Fleiss' κ + 95% CI (bootstrap)         ║
║    14  conditional_fr.csv            FR_correct / FR_incorrect               ║
║    15  summary_report.txt            Full plain-text digest                  ║
║    evaluation_results.zip            All files bundled for download          ║
║                                                                              ║
║  FIX v2: Fleiss kappa corrected — uses COUNT matrix not proportion matrix.  ║
║  Previous version produced κ outside [-1,+1]. Now κ ∈ [-1,+1] guaranteed.  ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess
pass  # pip install removed on purpose (avoids numpy upgrade)
print("✓ Dependencies ready")

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports & Config
# ═══════════════════════════════════════════════════════════════════════════════
import os, warnings, zipfile, pickle
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, spearmanr, pointbiserialr
from itertools import combinations

warnings.filterwarnings("ignore")
np.random.seed(42)

INPUT_CSV  = FIXED_CSV   # corrected file written by cell 4
OUT_DIR    = "cspf_evaluation_results"
N_BOOT     = 1000       # bootstrap iterations for CIs
ALPHA      = 0.05       # significance level
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_CSV} …")
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"  {len(df):,} rows  |  {df.shape[1]} columns")

MODELS     = sorted({c.replace("_flip","") for c in df.columns if c.endswith("_flip")})
FAMILIES   = sorted(df["perturbation_family"].dropna().unique())
SEVERITIES = ["mild","medium","strong"]
POSITIONS  = sorted(df["position_strategy"].dropna().unique()) if "position_strategy" in df.columns else []
DATASETS   = sorted(df["source_dataset"].dropna().unique()) if "source_dataset" in df.columns else []
GT_COL     = "sentiment"

FLIP_COLS  = [f"{m}_flip" for m in MODELS if f"{m}_flip" in df.columns]
ORIG_COLS  = [f"{m}_orig_label" for m in MODELS]
PERT_COLS  = [f"{m}_pert_label" for m in MODELS]

print(f"  Models ({len(MODELS)}): {MODELS}")
print(f"  Families: {len(FAMILIES)}  Datasets: {len(DATASETS)}  Positions: {len(POSITIONS)}")


# ═══════════════════════════════════════════════════════════════════════════════
# STATISTICAL HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def safe_mean(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.mean()) if len(s) else np.nan

def safe_std(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.std()) if len(s) > 1 else np.nan

def proportion_ci_wilson(n_success, n_total, alpha=ALPHA):
    """
    Wilson score confidence interval for a proportion.
    More accurate than normal approximation, especially for small n or extreme p.
    Returns (lower, upper).
    """
    if n_total == 0: return (np.nan, np.nan)
    z    = stats.norm.ppf(1 - alpha/2)
    p    = n_success / n_total
    denom = 1 + z**2 / n_total
    centre = (p + z**2 / (2*n_total)) / denom
    margin = z * np.sqrt(p*(1-p)/n_total + z**2/(4*n_total**2)) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))

def bootstrap_ci(values, stat_fn=np.mean, n_boot=N_BOOT, alpha=ALPHA):
    """
    Percentile bootstrap CI for any statistic.
    Returns (lower, upper, bootstrap_std).
    """
    values = np.array(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) < 2: return (np.nan, np.nan, np.nan)
    boots = [stat_fn(np.random.choice(values, size=len(values), replace=True))
             for _ in range(n_boot)]
    lo = np.percentile(boots, 100*alpha/2)
    hi = np.percentile(boots, 100*(1-alpha/2))
    return (round(lo,4), round(hi,4), round(np.std(boots),4))

def cohen_h(p1, p2):
    """Cohen's h for two proportions. h = 2·arcsin(√p₁) − 2·arcsin(√p₂)."""
    if np.isnan(p1) or np.isnan(p2): return np.nan
    return float(2*np.arcsin(np.sqrt(np.clip(p1,0,1))) -
                 2*np.arcsin(np.sqrt(np.clip(p2,0,1))))

def pages_L_test(fr_values_per_subject):
    """
    Page's L statistic for ordered alternatives.

    Correct implementation:
      - Each 'subject' is one perturbed review (or one review-model data point).
      - Conditions j ∈ {1,2,3} correspond to {mild, medium, strong}.
      - rᵢⱼ = rank of observation i under condition j (ranked within subjects).
      - L = Σⱼ j · (Σᵢ rᵢⱼ)

    fr_values_per_subject: list/array of shape (n, 3) — each row is
      [FR_at_mild, FR_at_medium, FR_at_strong] for one subject.

    Returns (L, z, p_one_tailed).
    """
    mat = np.array(fr_values_per_subject, dtype=float)
    if mat.ndim == 1:
        mat = mat.reshape(1, -1)
    mat = mat[~np.isnan(mat).any(axis=1)]  # drop rows with NaN
    n, k = mat.shape
    if n == 0 or k != 3: return (np.nan, np.nan, np.nan)

    # Rank within each subject (row)
    ranks = np.array([stats.rankdata(row) for row in mat])   # shape (n, k)
    # L statistic
    L = sum((j+1) * ranks[:, j].sum() for j in range(k))
    # Expected value and variance under H0 (no trend)
    mu    = n * k * (k+1)**2 / 4
    sigma = np.sqrt(n * k**2 * (k+1) * (k**2 - 1) / 144)
    z = (L - mu) / sigma if sigma > 0 else 0.0
    p = float(1 - stats.norm.cdf(z))   # one-tailed (H1: increasing)
    return (float(L), float(z), p)

def dunn_bonferroni(groups: dict):
    """
    Dunn's test with Bonferroni correction.
    groups: {name: array_like}
    Returns DataFrame[group1, group2, z, p_raw, p_bonferroni, significant].
    """
    names  = list(groups.keys())
    arrays = [np.array(v, dtype=float) for v in groups.values()]
    all_data  = np.concatenate(arrays)
    all_ranks = stats.rankdata(all_data)
    N = len(all_data); idx = 0
    ranks_per = []
    for a in arrays:
        ranks_per.append(all_ranks[idx:idx+len(a)]); idx += len(a)
    pairs = list(combinations(range(len(names)), 2))
    k = len(pairs); rows = []
    for i, j in pairs:
        ni, nj = len(arrays[i]), len(arrays[j])
        Ri, Rj = ranks_per[i].mean(), ranks_per[j].mean()
        se = np.sqrt((N*(N+1)/12)*(1/ni + 1/nj))
        z  = (Ri - Rj) / se if se > 0 else 0.0
        p_raw  = float(2*(1 - stats.norm.cdf(abs(z))))
        p_bonf = min(p_raw * k, 1.0)
        rows.append({"group1": names[i], "group2": names[j],
                     "z": round(z,4), "p_raw": round(p_raw,6),
                     "p_bonferroni": round(p_bonf,6),
                     f"significant_{ALPHA}": p_bonf < ALPHA})
    return pd.DataFrame(rows)

def fleiss_kappa_with_ci(flip_matrix, n_boot=N_BOOT, alpha=ALPHA):
    """
    Fleiss' kappa + bootstrap 95% CI.
    flip_matrix : (n_items, n_raters) binary array.

    FIX: uses COUNT matrix C[i,j] = number of raters assigning item i to
    category j. Previous version passed proportions to a count-based formula
    producing kappa outside [-1,+1]. Now correctly builds:
      C[:,0] = n_raters - flip_sum_per_item
      C[:,1] = flip_sum_per_item
    Returns (kappa, z, p, ci_lo, ci_hi).
    """
    def _kappa(count_mat):
        count_mat = np.array(count_mat, dtype=float)
        n, k = count_mat.shape
        N = count_mat.sum(axis=1)
        N_bar = N.mean()
        if N_bar <= 1: return 0.0
        pj = count_mat.sum(axis=0) / (n * N_bar)
        Pe = (pj ** 2).sum()
        Pi = ((count_mat ** 2).sum(axis=1) - N) / (N * (N - 1))
        Pbar = Pi.mean()
        denom = 1 - Pe
        return float((Pbar - Pe) / denom) if denom != 0 else 0.0

    mat = np.array(flip_matrix, dtype=float)
    n_items, n_raters = mat.shape
    n_flip1  = mat.sum(axis=1)
    n_flip0  = n_raters - n_flip1
    count_mat = np.stack([n_flip0, n_flip1], axis=1)

    kappa = _kappa(count_mat)
    pj    = count_mat.sum(axis=0) / (n_items * n_raters)
    Pe    = (pj ** 2).sum()
    se    = np.sqrt(2 / (n_items * n_raters * (n_raters - 1))) if n_items > 1 else 0.0
    z     = kappa / se if se > 0 else 0.0
    p     = float(2 * (1 - stats.norm.cdf(abs(z))))

    boots = []
    for _ in range(n_boot):
        idx = np.random.choice(n_items, size=n_items, replace=True)
        boots.append(_kappa(count_mat[idx]))
    ci_lo = float(np.percentile(boots, 100 * alpha / 2))
    ci_hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return (round(kappa,4), round(z,4), round(p,6), round(ci_lo,4), round(ci_hi,4))


# ═══════════════════════════════════════════════════════════════════════════════
# 01 — FLIP RATE MATRIX  FR(family × severity × model) + 95% Wilson CI
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[01] Flip Rate Matrix + Wilson CIs …")

rows_01 = []
for fam in FAMILIES:
    for sev in SEVERITIES:
        mask = (df["perturbation_family"]==fam) & (df["severity"]==sev)
        sub  = df[mask]
        for model in MODELS:
            fc = f"{model}_flip"
            if fc not in sub.columns: continue
            vals = sub[fc].dropna()
            n_total   = len(vals)
            n_flips   = int(vals.sum())
            fr        = n_flips / n_total if n_total > 0 else np.nan
            ci_lo, ci_hi = proportion_ci_wilson(n_flips, n_total)
            rows_01.append({
                "perturbation_family": fam, "severity": sev, "model": model,
                "n_pairs": n_total, "n_flips": n_flips,
                "flip_rate": round(fr,4) if not np.isnan(fr) else np.nan,
                "CI_lower_95": round(ci_lo,4), "CI_upper_95": round(ci_hi,4),
                "CI_width": round(ci_hi - ci_lo, 4),
            })

fr_df = pd.DataFrame(rows_01)
fr_df.to_csv(f"{OUT_DIR}/01_flip_rate_matrix.csv", index=False)
print(f"  ✓ {len(fr_df):,} rows  (FR range: "
      f"{fr_df['flip_rate'].min():.4f} – {fr_df['flip_rate'].max():.4f})")


# ═══════════════════════════════════════════════════════════════════════════════
# 02 — MODEL ROBUSTNESS SCORES + Bootstrap CI
# RS, RS_conditional, DR, IRR, acc_original
# ═══════════════════════════════════════════════════════════════════════════════
print("[02] Model Robustness Scores + Bootstrap CIs …")

rows_02 = []
for model in MODELS:
    fc = f"{model}_flip"; oc = f"{model}_orig_label"; pc = f"{model}_pert_label"
    if fc not in df.columns: continue

    flip_vals = df[fc].dropna().values
    RS        = float(flip_vals.mean()) if len(flip_vals) else np.nan
    rs_lo, rs_hi, _ = bootstrap_ci(flip_vals)

    # Accuracy on original (deduplicated by review_idx)
    if "review_key" in df.columns and oc in df.columns and GT_COL in df.columns:
        unique = df.drop_duplicates("review_key")   # FIX: composite key (dataset::review_idx)
        acc    = float((unique[oc] == unique[GT_COL]).mean())
    elif oc in df.columns and GT_COL in df.columns:
        acc = float((df[oc] == df[GT_COL]).mean())
    else:
        acc = np.nan

    # Conditional: only correctly classified originals
    if oc in df.columns and GT_COL in df.columns:
        correct_mask = df[oc] == df[GT_COL]
        RS_c = safe_mean(df.loc[correct_mask, fc])
        rc_lo, rc_hi, _ = bootstrap_ci(df.loc[correct_mask, fc].dropna().values)
    else:
        RS_c = rc_lo = rc_hi = np.nan

    # DR: correct original → wrong perturbed
    if oc in df.columns and pc in df.columns and GT_COL in df.columns:
        dr_mask = (df[oc] == df[GT_COL]) & (df[pc] != df[GT_COL])
        DR      = float(dr_mask.mean())
        dr_lo, dr_hi, _ = bootstrap_ci(dr_mask.astype(float).values)
    else:
        DR = dr_lo = dr_hi = np.nan

    # IRR: wrong original → correct perturbed
    if oc in df.columns and pc in df.columns and GT_COL in df.columns:
        irr_mask = (df[oc] != df[GT_COL]) & (df[pc] == df[GT_COL])
        IRR      = float(irr_mask.mean())
    else:
        IRR = np.nan

    rows_02.append({
        "model": model,
        "RS":              round(RS, 4),
        "RS_CI_lo":        rs_lo, "RS_CI_hi": rs_hi,
        "RS_conditional":  round(RS_c,4) if not np.isnan(RS_c) else np.nan,
        "RS_c_CI_lo":      rc_lo, "RS_c_CI_hi": rc_hi,
        "DR_degradation":  round(DR,4)  if not np.isnan(DR)  else np.nan,
        "DR_CI_lo":        dr_lo, "DR_CI_hi": dr_hi,
        "IRR_improvement": round(IRR,4) if not np.isnan(IRR) else np.nan,
        "acc_original":    round(acc,4) if not np.isnan(acc)  else np.nan,
        "n_total_pairs":   int(df[fc].notna().sum()),
        "n_flips_total":   int(df[fc].sum()),
    })

rs_df = pd.DataFrame(rows_02).sort_values("RS")
rs_df.to_csv(f"{OUT_DIR}/02_model_robustness_scores.csv", index=False)
print("  Robustness ranking (lower RS = more robust):")
for _, r in rs_df.iterrows():
    print(f"    {r['model']:12s}  RS={r['RS']:.4f} [{r['RS_CI_lo']:.4f},{r['RS_CI_hi']:.4f}]  "
          f"DR={r['DR_degradation']:.4f}  acc={r['acc_original']:.4f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 03 — SEVERITY MONOTONICITY  Page's L (correct implementation)
#
# For each (family, model) pair we build the per-review FR matrix:
#   rows = unique reviews that appear in all 3 severity levels for this family
#   cols = [FR_at_mild, FR_at_medium, FR_at_strong]  (binary: 0 or 1 per review)
# Then apply Page's L across these n subjects.
# ═══════════════════════════════════════════════════════════════════════════════
print("[03] Severity Monotonicity — Page's L (per-review subject matrix) …")

rows_03 = []
for fam in FAMILIES:
    sub_fam = df[df["perturbation_family"]==fam]
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in sub_fam.columns: continue

        # Build per-review flip indicator across severity levels
        sev_data = {}
        for sev in SEVERITIES:
            sev_sub = sub_fam[sub_fam["severity"]==sev][["review_idx", fc]].dropna() \
                if "review_idx" in df.columns \
                else sub_fam[sub_fam["severity"]==sev][[fc]].dropna()
            if "review_idx" in df.columns:
                # One value per review (take mean in case of multiple variants)
                sev_data[sev] = sev_sub.groupby("review_idx")[fc].mean()
            else:
                sev_data[sev] = sev_sub[fc].reset_index(drop=True)

        # Aggregate FR per severity as the subject-level matrix
        if "review_idx" in df.columns:
            common_idx = set(sev_data["mild"].index)
            for sev in SEVERITIES[1:]:
                common_idx &= set(sev_data[sev].index)
            common_idx = sorted(common_idx)
            if len(common_idx) < 3:
                # Fallback: use aggregate FR per severity as a single-row matrix
                fr_vec = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
                subjects_matrix = [fr_vec]
            else:
                subjects_matrix = [[sev_data[s].loc[i] for s in SEVERITIES]
                                   for i in common_idx]
        else:
            fr_vec = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
            subjects_matrix = [fr_vec]

        L, z_page, p_page = pages_L_test(subjects_matrix)

        fr_vals = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
        rho, p_spearman = spearmanr([1,2,3], fr_vals) if not any(np.isnan(fr_vals)) else (np.nan, np.nan)

        rows_03.append({
            "perturbation_family": fam, "model": model,
            "FR_mild":   round(fr_vals[0],4) if not np.isnan(fr_vals[0]) else np.nan,
            "FR_medium": round(fr_vals[1],4) if not np.isnan(fr_vals[1]) else np.nan,
            "FR_strong": round(fr_vals[2],4) if not np.isnan(fr_vals[2]) else np.nan,
            "monotone_FR": (fr_vals[0]<=fr_vals[1]<=fr_vals[2])
                            if not any(np.isnan(fr_vals)) else None,
            "n_subjects":  len(subjects_matrix),
            "pages_L":     round(L,2)        if not np.isnan(L)         else np.nan,
            "pages_z":     round(z_page,4)   if not np.isnan(z_page)    else np.nan,
            "pages_p":     round(p_page,6)   if not np.isnan(p_page)    else np.nan,
            "pages_sig":   (p_page < ALPHA)  if not np.isnan(p_page)    else None,
            "spearman_rho":round(rho,4)      if not np.isnan(rho)       else np.nan,
            "spearman_p":  round(p_spearman,6) if not np.isnan(p_spearman) else np.nan,
        })

mono_df = pd.DataFrame(rows_03)
mono_df.to_csv(f"{OUT_DIR}/03_severity_monotonicity.csv", index=False)
n_mono = mono_df["monotone_FR"].sum() if "monotone_FR" in mono_df else 0
n_sig  = mono_df["pages_sig"].sum()   if "pages_sig"  in mono_df else 0
print(f"  Monotone pairs  : {n_mono}/{len(mono_df)}")
print(f"  Page's L sig    : {n_sig}/{len(mono_df)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 04 — POSITION STRATEGY ANALYSIS + Kruskal-Wallis across positions
# ═══════════════════════════════════════════════════════════════════════════════
print("[04] Position Strategy Analysis …")

rows_04 = []
for model in MODELS:
    fc = f"{model}_flip"
    if fc not in df.columns: continue
    groups_pos = {}
    for pos in POSITIONS:
        sub = df[df["position_strategy"]==pos]
        vals = sub[fc].dropna().values
        if len(vals) > 0:
            fr = float(vals.mean())
            n  = len(vals); n_flip = int(vals.sum())
            ci_lo, ci_hi = proportion_ci_wilson(n_flip, n)
            rows_04.append({
                "model": model, "position_strategy": pos,
                "n_pairs": n, "n_flips": n_flip,
                "flip_rate": round(fr,4),
                "CI_lower_95": round(ci_lo,4), "CI_upper_95": round(ci_hi,4),
            })
            groups_pos[pos] = vals
    # Kruskal-Wallis across positions for this model
    if len(groups_pos) >= 2:
        H, p = kruskal(*groups_pos.values())
        rows_04.append({
            "model": model, "position_strategy": "__Kruskal_H__",
            "n_pairs": sum(len(v) for v in groups_pos.values()),
            "n_flips": np.nan, "flip_rate": round(H,4),
            "CI_lower_95": round(p,6), "CI_upper_95": len(groups_pos)-1,
        })

pos_df = pd.DataFrame(rows_04)
pos_df.to_csv(f"{OUT_DIR}/04_position_analysis.csv", index=False)
print(f"  ✓ {len(POSITIONS)} positions × {len(MODELS)} models")


# ═══════════════════════════════════════════════════════════════════════════════
# 05 — DATASET VULNERABILITY INDEX  DVI
# ═══════════════════════════════════════════════════════════════════════════════
print("[05] Dataset Vulnerability Index …")

rows_05 = []
for ds in DATASETS:
    sub_ds = df[df["source_dataset"]==ds] if "source_dataset" in df.columns else df
    model_frs = {}
    for model in MODELS:
        fc = f"{model}_flip"
        if fc in sub_ds.columns:
            v = sub_ds[fc].dropna().values
            if len(v): model_frs[model] = float(v.mean())
    dvi = np.mean(list(model_frs.values())) if model_frs else np.nan
    row = {"source_dataset": ds,
           "DVI": round(dvi,4) if not np.isnan(dvi) else np.nan,
           "n_pairs": len(sub_ds)}
    row.update({f"FR_{m}": round(model_frs.get(m, np.nan),4) for m in MODELS})
    rows_05.append(row)

dvi_df = pd.DataFrame(rows_05).sort_values("DVI", ascending=False)
dvi_df.to_csv(f"{OUT_DIR}/05_dataset_vulnerability.csv", index=False)
print(f"  ✓ {len(rows_05)} datasets  |  DVI range: "
      f"{dvi_df['DVI'].min():.4f} – {dvi_df['DVI'].max():.4f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 06 — KRUSKAL-WALLIS + DUNN POST-HOC
# ═══════════════════════════════════════════════════════════════════════════════
print("[06] Kruskal-Wallis + Dunn post-hoc …")

rows_kw = []; dunn_all = []
for fam in FAMILIES:
    sub_f  = df[df["perturbation_family"]==fam]
    groups = {m: sub_f[f"{m}_flip"].dropna().values
              for m in MODELS if f"{m}_flip" in sub_f.columns
              and len(sub_f[f"{m}_flip"].dropna()) > 0}
    if len(groups) < 2: continue
    H, p = kruskal(*groups.values())
    eta2 = (H - len(groups) + 1) / (len(df[df["perturbation_family"]==fam]) - len(groups)) \
           if (len(df[df["perturbation_family"]==fam]) - len(groups)) > 0 else np.nan
    rows_kw.append({
        "perturbation_family": fam, "kruskal_H": round(H,4),
        "df": len(groups)-1, "kruskal_p": round(p,6),
        "eta_squared": round(eta2,4) if not np.isnan(eta2) else np.nan,
        f"significant_{ALPHA}": p < ALPHA,
    })
    dunn_res = dunn_bonferroni(groups)
    dunn_res["perturbation_family"] = fam
    dunn_all.append(dunn_res)

kw_df = pd.DataFrame(rows_kw)
kw_df.to_csv(f"{OUT_DIR}/06a_kruskal_wallis.csv", index=False)
n_sig_kw = kw_df[f"significant_{ALPHA}"].sum() if len(kw_df) else 0
print(f"  Kruskal-Wallis: {n_sig_kw}/{len(kw_df)} families significant")

if dunn_all:
    dunn_df = pd.concat(dunn_all, ignore_index=True)
    dunn_df.to_csv(f"{OUT_DIR}/06b_dunn_posthoc.csv", index=False)
    n_sig_d = dunn_df[f"significant_{ALPHA}"].sum()
    print(f"  Dunn sig pairs (Bonferroni): {n_sig_d}/{len(dunn_df)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 07 — PIR TENSOR  [12 × 3 × 3 × 9]
# ═══════════════════════════════════════════════════════════════════════════════
print("[07] Building PIR tensor …")

n_F,n_S,n_P,n_M = len(FAMILIES),len(SEVERITIES),max(len(POSITIONS),1),len(MODELS)
pir = np.full((n_F,n_S,n_P,n_M), np.nan)

_pos_list = POSITIONS if POSITIONS else ["ALL"]
for fi,fam in enumerate(FAMILIES):
    for si,sev in enumerate(SEVERITIES):
        for pi,pos in enumerate(_pos_list):
            if POSITIONS:
                mask = ((df["perturbation_family"]==fam) &
                        (df["severity"]==sev) &
                        (df["position_strategy"]==pos))
            else:
                mask = (df["perturbation_family"]==fam) & (df["severity"]==sev)
            sub = df[mask]
            if len(sub)==0: continue
            for mi,model in enumerate(MODELS):
                fc = f"{model}_flip"
                if fc in sub.columns:
                    v = sub[fc].dropna()
                    if len(v): pir[fi,si,pi,mi] = float(v.mean())

with open(f"{OUT_DIR}/07_pir_tensor.pkl","wb") as f:
    pickle.dump({"tensor":pir,
                 "dims":{"0_families":FAMILIES,"1_severities":SEVERITIES,
                         "2_positions":_pos_list,"3_models":MODELS}}, f)
print(f"  Tensor shape: {pir.shape}  |  Non-NaN: {(~np.isnan(pir)).sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# 08 — EFFECT SIZES  Cohen's h + Bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[08] Cohen's h effect sizes + Bootstrap CIs …")

rows_08 = []
for fam in FAMILIES:
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in df.columns: continue
        sub_mild   = df[(df["perturbation_family"]==fam)&(df["severity"]=="mild")][fc].dropna()
        sub_strong = df[(df["perturbation_family"]==fam)&(df["severity"]=="strong")][fc].dropna()
        if len(sub_mild)==0 or len(sub_strong)==0: continue

        p_mild   = float(sub_mild.mean())
        p_strong = float(sub_strong.mean())
        h = cohen_h(p_strong, p_mild)

        # Bootstrap CI for Cohen's h
        boots_h = []
        for _ in range(N_BOOT):
            bm = np.random.choice(sub_mild.values,   size=len(sub_mild),   replace=True).mean()
            bs = np.random.choice(sub_strong.values, size=len(sub_strong), replace=True).mean()
            boots_h.append(cohen_h(bs, bm))
        boots_h = [v for v in boots_h if not np.isnan(v)]
        h_lo = float(np.percentile(boots_h, 100*ALPHA/2))   if boots_h else np.nan
        h_hi = float(np.percentile(boots_h, 100*(1-ALPHA/2))) if boots_h else np.nan

        interp = "large" if abs(h)>=0.8 else ("medium" if abs(h)>=0.5 else "small")
        rows_08.append({
            "perturbation_family": fam, "model": model,
            "FR_mild": round(p_mild,4), "FR_strong": round(p_strong,4),
            "cohen_h": round(h,4) if not np.isnan(h) else np.nan,
            "cohen_h_CI_lo": round(h_lo,4), "cohen_h_CI_hi": round(h_hi,4),
            "effect_size": interp,
        })

eff_df = pd.DataFrame(rows_08)
eff_df.to_csv(f"{OUT_DIR}/08_effect_sizes.csv", index=False)
print(f"  large={( eff_df['effect_size']=='large').sum()}  "
      f"medium={( eff_df['effect_size']=='medium').sum()}  "
      f"small={( eff_df['effect_size']=='small').sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# 09 — SEMANTIC PRESERVATION ↔ FLIP  Point-biserial r + Spearman ρ + CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[09] Semantic × flip correlations …")

rows_09 = []
sem_col = "sem_preservation_score"
if sem_col in df.columns:
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in df.columns: continue
        sub = df[[sem_col, fc]].dropna()
        if len(sub) < 10: continue

        r_pb, p_pb = pointbiserialr(sub[fc].astype(int), sub[sem_col])
        rho, p_rho = spearmanr(sub[sem_col], sub[fc])

        # Bootstrap CI for r_pb
        boots = []
        arr = sub.values
        for _ in range(N_BOOT):
            idx = np.random.choice(len(arr), size=len(arr), replace=True)
            sample = arr[idx]
            try:
                r_b,_ = pointbiserialr(sample[:,1].astype(int), sample[:,0])
                boots.append(r_b)
            except: pass
        boots = [v for v in boots if not np.isnan(v)]
        r_lo = float(np.percentile(boots, 100*ALPHA/2))   if boots else np.nan
        r_hi = float(np.percentile(boots, 100*(1-ALPHA/2))) if boots else np.nan

        rows_09.append({
            "model": model, "n": len(sub),
            "r_pb": round(r_pb,4), "r_pb_CI_lo": round(r_lo,4), "r_pb_CI_hi": round(r_hi,4),
            "p_pb": round(p_pb,6),
            "spearman_rho": round(rho,4), "spearman_p": round(p_rho,6),
            "direction": "expected (higher Φ→fewer flips)" if r_pb<0 else "unexpected",
        })

sem_df = pd.DataFrame(rows_09)
sem_df.to_csv(f"{OUT_DIR}/09_semantic_vs_flip.csv", index=False)
print(f"  ✓ Mean r_pb = {sem_df['r_pb'].mean():.4f}  "
      f"({'negative = expected' if sem_df['r_pb'].mean()<0 else 'positive = unexpected'})")


# ═══════════════════════════════════════════════════════════════════════════════
# 10 — SEMANTIC METRIC DRIFT ACROSS SEVERITY
# ═══════════════════════════════════════════════════════════════════════════════
print("[10] Metric drift across severity …")

rows_10 = []
metric_cols = [c for c in ["m1_cosine_sim","m2_rougeL","m3_nli_o2p","m3_nli_p2o",
                             "m4_srp","m5_ppl_ratio","m6_edit_dist","m7_lsd","m8_pqs"]
               if c in df.columns]
for mc in metric_cols:
    for fam in FAMILIES:
        for sev in SEVERITIES:
            sub = df[(df["perturbation_family"]==fam)&(df["severity"]==sev)][mc].dropna()
            if len(sub)==0: continue
            rows_10.append({
                "metric": mc, "perturbation_family": fam, "severity": sev,
                "n": len(sub), "mean": round(sub.mean(),4), "std": round(sub.std(),4),
                "median": round(sub.median(),4),
                "q25": round(sub.quantile(0.25),4), "q75": round(sub.quantile(0.75),4),
            })

pd.DataFrame(rows_10).to_csv(f"{OUT_DIR}/10_severity_drift.csv", index=False)
print(f"  ✓ {len(rows_10)} metric-family-severity cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 11 — LINGUISTIC TARGET FR
# ═══════════════════════════════════════════════════════════════════════════════
print("[11] Linguistic target analysis …")
rows_11 = []
if "linguistic_target" in df.columns:
    for lt in sorted(df["linguistic_target"].dropna().unique()):
        sub = df[df["linguistic_target"]==lt]
        for model in MODELS:
            fc = f"{model}_flip"
            if fc not in sub.columns: continue
            vals = sub[fc].dropna()
            if len(vals)==0: continue
            n=len(vals); nf=int(vals.sum())
            ci_lo,ci_hi = proportion_ci_wilson(nf,n)
            rows_11.append({"linguistic_target":lt,"model":model,"n":n,"n_flips":nf,
                             "flip_rate":round(float(vals.mean()),4),
                             "CI_lower_95":round(ci_lo,4),"CI_upper_95":round(ci_hi,4)})
    pd.DataFrame(rows_11).to_csv(f"{OUT_DIR}/11_linguistic_target_fr.csv", index=False)
    print(f"  ✓ {df['linguistic_target'].nunique()} targets")


# ═══════════════════════════════════════════════════════════════════════════════
# 12 — CROSS-DATASET × FAMILY FR HEATMAP DATA
# ═══════════════════════════════════════════════════════════════════════════════
print("[12] Cross-dataset FR heatmap …")
rows_12 = []
if DATASETS:
    for ds in DATASETS:
        sub_ds = df[df["source_dataset"]==ds]
        for fam in FAMILIES:
            sub = sub_ds[sub_ds["perturbation_family"]==fam]
            vals = [safe_mean(sub[f"{m}_flip"]) for m in MODELS
                    if f"{m}_flip" in sub.columns]
            vals = [v for v in vals if not np.isnan(v)]
            mfr = round(np.mean(vals),4) if vals else np.nan
            rows_12.append({"source_dataset":ds,"perturbation_family":fam,
                             "mean_FR_all_models":mfr,"n":len(sub)})
    pd.DataFrame(rows_12).to_csv(f"{OUT_DIR}/12_cross_dataset_fr.csv", index=False)
    print(f"  ✓ {len(rows_12)} dataset-family cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 13 — INTER-MODEL AGREEMENT  Fleiss' κ + Bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[13] Inter-model agreement (Fleiss' κ) + Bootstrap CI …")

rows_13 = []
flip_avail = [f"{m}_flip" for m in MODELS if f"{m}_flip" in df.columns]

if len(flip_avail) >= 2:
    flip_mat = df[flip_avail].fillna(0).astype(int).values  # (n_items, n_raters)

    # Overall
    k,z_k,p_k,ci_lo,ci_hi = fleiss_kappa_with_ci(flip_mat)
    interp = ("almost perfect" if k>0.80 else "substantial" if k>0.60
              else "moderate" if k>0.40 else "fair" if k>0.20 else "poor")
    rows_13.append({"scope":"overall","perturbation_family":"ALL",
                    "fleiss_kappa":k,"z":z_k,"p":p_k,
                    "CI_lo_95":ci_lo,"CI_hi_95":ci_hi,
                    "n_raters":flip_mat.shape[1],"n_items":flip_mat.shape[0],
                    "interpretation":interp})

    # Per family
    for fam in FAMILIES:
        mask  = df["perturbation_family"]==fam
        f_mat = df.loc[mask, flip_avail].fillna(0).astype(int).values
        if f_mat.shape[0] < 5: continue
        k2,z2,p2,l2,h2 = fleiss_kappa_with_ci(f_mat, n_boot=200)
        interp2 = ("almost perfect" if k2>0.80 else "substantial" if k2>0.60
                   else "moderate" if k2>0.40 else "fair" if k2>0.20 else "poor")
        rows_13.append({"scope":"family","perturbation_family":fam,
                        "fleiss_kappa":k2,"z":z2,"p":p2,
                        "CI_lo_95":l2,"CI_hi_95":h2,
                        "n_raters":f_mat.shape[1],"n_items":f_mat.shape[0],
                        "interpretation":interp2})

agree_df = pd.DataFrame(rows_13)
agree_df.to_csv(f"{OUT_DIR}/13_agreement_matrix.csv", index=False)
ov = agree_df[agree_df["scope"]=="overall"]
if len(ov):
    r = ov.iloc[0]
    print(f"  Overall κ = {r['fleiss_kappa']:.4f}  "
          f"[{r['CI_lo_95']:.4f}, {r['CI_hi_95']:.4f}]  ({r['interpretation']})")


# ═══════════════════════════════════════════════════════════════════════════════
# 14 — CONDITIONAL FLIP RATE  FR_correct vs FR_incorrect
# ═══════════════════════════════════════════════════════════════════════════════
print("[14] Conditional Flip Rate …")

rows_14 = []
for model in MODELS:
    fc=f"{model}_flip"; oc=f"{model}_orig_label"
    if fc not in df.columns or oc not in df.columns or GT_COL not in df.columns: continue
    correct = df[oc]==df[GT_COL]
    for fam in FAMILIES:
        fam_mask = df["perturbation_family"]==fam
        for cond_lbl, mask in [("correct_original", correct & fam_mask),
                                ("incorrect_original", ~correct & fam_mask)]:
            sub = df.loc[mask, fc].dropna()
            n=len(sub); nf=int(sub.sum())
            ci_lo,ci_hi = proportion_ci_wilson(nf,n)
            rows_14.append({
                "model":model,"perturbation_family":fam,"condition":cond_lbl,
                "n":n,"n_flips":nf,
                "flip_rate":round(float(sub.mean()),4) if n>0 else np.nan,
                "CI_lower_95":round(ci_lo,4),"CI_upper_95":round(ci_hi,4),
            })

pd.DataFrame(rows_14).to_csv(f"{OUT_DIR}/14_conditional_fr.csv", index=False)
print(f"  ✓ {len(rows_14)} model-family-condition cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 15 — SUMMARY REPORT
# ═══════════════════════════════════════════════════════════════════════════════
print("[15] Writing summary report …")

SEP  = "═"*72
sep2 = "─"*72

lines = [SEP,
"  CSPF PHASE-3 EVALUATION — COMPLETE SUMMARY REPORT",
f"  {len(df):,} pairs  |  {len(MODELS)} models  |  {len(FAMILIES)} families  |  "
f"n_boot={N_BOOT}  α={ALPHA}",
SEP,""]

# Robustness ranking
lines += ["1. MODEL ROBUSTNESS RANKING  (RS ± 95% Bootstrap CI)", sep2]
for _,r in rs_df.iterrows():
    lines.append(f"  {r['model']:12s}  RS={r['RS']:.4f} [{r['RS_CI_lo']:.4f},{r['RS_CI_hi']:.4f}]  "
                 f"DR={r['DR_degradation']:.4f} [{r['DR_CI_lo']:.4f},{r['DR_CI_hi']:.4f}]  "
                 f"acc={r['acc_original']:.4f}")

# FR by family
lines += ["","2. FLIP RATE BY FAMILY (mean across models × severities)", sep2]
fam_fr = fr_df.groupby("perturbation_family")["flip_rate"].mean().sort_values(ascending=False)
for fam,fr in fam_fr.items():
    lines.append(f"  {fam:38s}  FR={fr:.4f}")

# Monotonicity
lines += ["","3. SEVERITY MONOTONICITY (Page's L)", sep2]
if len(mono_df):
    n_valid = mono_df["monotone_FR"].notna().sum()
    pct = 100*mono_df["monotone_FR"].sum()/max(n_valid,1)
    n_sig_p = mono_df["pages_sig"].sum()
    lines += [f"  Monotone pairs       : {int(mono_df['monotone_FR'].sum())}/{n_valid}  ({pct:.1f}%)",
              f"  Page's L significant : {n_sig_p}/{len(mono_df)}  (p < {ALPHA})"]

# Kruskal-Wallis
lines += ["","4. KRUSKAL-WALLIS (cross-model FR per family)", sep2]
if len(kw_df):
    lines.append(f"  Significant: {n_sig_kw}/{len(kw_df)} families  (α = {ALPHA}, Bonf-corrected pairwise)")

# Effect sizes
lines += ["","5. EFFECT SIZES (Cohen's h: FR_strong vs FR_mild)", sep2]
if len(eff_df):
    for cat in ["large","medium","small"]:
        n = (eff_df["effect_size"]==cat).sum()
        lines.append(f"  {cat:6s}: {n:3d} family-model pairs")

# Fleiss kappa
lines += ["","6. INTER-MODEL AGREEMENT (Fleiss' κ)", sep2]
if len(agree_df):
    r = agree_df[agree_df["scope"]=="overall"].iloc[0]
    lines.append(f"  κ = {r['fleiss_kappa']:.4f}  95% CI [{r['CI_lo_95']:.4f}, {r['CI_hi_95']:.4f}]  "
                 f"({r['interpretation']})")

# Semantic correlation
lines += ["","7. SEMANTIC PRESERVATION ↔ FLIP CORRELATION (r_pb + 95% CI)", sep2]
if len(sem_df):
    for _,r in sem_df.iterrows():
        lines.append(f"  {r['model']:12s}  r_pb={r['r_pb']:.4f} [{r['r_pb_CI_lo']:.4f},{r['r_pb_CI_hi']:.4f}]  "
                     f"ρ={r['spearman_rho']:.4f}")

# DVI
lines += ["","8. DATASET VULNERABILITY (top 5)", sep2]
if len(dvi_df):
    for _,r in dvi_df.head(5).iterrows():
        lines.append(f"  {str(r['source_dataset']):30s}  DVI={r['DVI']:.4f}  n={r['n_pairs']:,}")

lines += ["", SEP,
"  FILES PRODUCED",sep2,
"  01  flip_rate_matrix.csv          FR(family×severity×model) + Wilson 95% CI",
"  02  model_robustness_scores.csv   RS, RS_c, DR, IRR, acc + Bootstrap 95% CI",
"  03  severity_monotonicity.csv     Page's L (per-review), Spearman ρ",
"  04  position_analysis.csv         FR by position + Kruskal-Wallis",
"  05  dataset_vulnerability.csv     DVI + per-model FR per dataset",
"  06a kruskal_wallis.csv            H, df, p, η² per family",
"  06b dunn_posthoc.csv              Dunn z / p_bonferroni per pair",
"  07  pir_tensor.pkl                4D PIR tensor [F×S×P×M]",
"  08  effect_sizes.csv              Cohen's h + Bootstrap 95% CI",
"  09  semantic_vs_flip.csv          r_pb + Bootstrap CI + Spearman ρ",
"  10  severity_drift.csv            Metric means/std/Q1/Q3 per (metric,fam,sev)",
"  11  linguistic_target_fr.csv      FR per linguistic target + CI",
"  12  cross_dataset_fr.csv          FR heatmap (family × dataset)",
"  13  agreement_matrix.csv          Fleiss' κ + Bootstrap 95% CI overall+family",
"  14  conditional_fr.csv            FR_correct / FR_incorrect per (model,family)+CI",
"  15  summary_report.txt            This file",
SEP,""]

report = "\n".join(lines)
with open(f"{OUT_DIR}/15_summary_report.txt","w") as f: f.write(report)
print(report)

# ═══════════════════════════════════════════════════════════════════════════════
# ZIP + DOWNLOAD
# ═══════════════════════════════════════════════════════════════════════════════
zip_path = "evaluation_results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for root,_,files in os.walk(OUT_DIR):
        for fname in files:
            fpath = os.path.join(root,fname)
            zf.write(fpath, os.path.relpath(fpath,OUT_DIR))
print(f"\n✓ ZIP: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Download triggered.")
except ImportError:
    print(f"  (Not in Colab — {zip_path} is in the working directory.)")

In [ ]:
# ── small bundle to send back (a few KB) ──────────────────────────────────
import glob, zipfile
with zipfile.ZipFile("fix_tables.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("cspf_evaluation_results/*.csv")+["alignment_check.csv","extra_models_registry.csv","per_dataset_orig_accuracy.csv","cspf_evaluation_results/15_summary_report.txt"]:
        if os.path.exists(f) and os.path.getsize(f)<5_000_000: z.write(f)
print(os.path.getsize("fix_tables.zip")/1e3,"KB")
try:
    from google.colab import files; files.download("fix_tables.zip")
except Exception as e: print(e)
print(open("alignment_check.csv").read())

In [ ]:
# ── copy ALL results to Drive (so nothing is lost when Colab disconnects) ──
import shutil, os
DEST = "/content/drive/MyDrive/postdoc-papers/paper2/results_FIXED"; os.makedirs(DEST, exist_ok=True)
shutil.copytree("cspf_evaluation_results", DEST + "/cspf_evaluation_results", dirs_exist_ok=True)
for f in ["alignment_check.csv","per_dataset_orig_accuracy.csv","extra_models_registry.csv","fix_tables.zip","cspf_results_final.zip"]:
    if os.path.exists(f): shutil.copy(f, DEST)
print("saved to Drive:", DEST); print(os.listdir(DEST))
